# Salt wedge benchmark: FiPy backend versus experiment

This notebook reproduces the comparison of the GroMPy FiPy backend with the salt
wedge intrusion experiments of Goswami and Clement (2007, WRR 43). It runs the three
steady state scenarios with the FiPy backend, extracts the modeled 0.5 isochlor,
computes the misfit against the measured salt wedge, and generates a figure in the
same style as `benchmark_data/model_vs_experimental_salt_wedge.png`.

The FiPy backend uses an orthogonal Grid2D for the rectangular domain, a sequential
(Picard) coupling of the flow and transport equations with under-relaxation, and the
full anisotropic dispersion tensor.

Note: running the three scenarios to steady state takes roughly 15 to 20 minutes. The
results are cached to a file so the plotting cells can be re-run without recomputing.

## Imports

In [ ]:
import os
import sys
import time
import pickle
import warnings
from contextlib import redirect_stdout

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# make sure the repository root is importable when the notebook lives there
sys.path.insert(0, os.path.abspath('.'))

import lib.read_vtu_file
from model_input.model_parameters_sw_benchmark import (
    ModelParameters, ModelOptions, ParameterRanges)
from lib.grompy_fipy import run_coupled_flow_model_fipy

%matplotlib inline

## Model run configuration

The settings below are the validated FiPy backend configuration. The 5 mm cell size is
mesh converged for the wedge toe (a 2.5 mm mesh gives the same result at four times the
cost). The coupling is under-relaxed at 0.5 so the Picard iteration converges each
timestep, which is essential for the deep, buoyancy dominated SS-2 wedge. The simulation
time is set to 8 hours of model time so that the deep wedge reaches steady state.

In [ ]:
CELLSIZE = 0.005
DT_MAX = 10.0
RELAX = 0.5
MAX_ITER = 15
TOTAL_TIME = 8.0 * 60.0 * 60.0

MAX_CONC = 0.03624
ISOCHLOR_FRACTIONS = [0.1, 0.5, 0.9]

DATA_DIR = 'benchmark_data'
OUTPUT_DIR = 'model_output/salt_wedge_benchmark_fipy'
CACHE_FILE = 'model_output/salt_wedge_benchmark_fipy/notebook_isochlors.pkl'
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Run one salt wedge scenario with the FiPy backend

The three scenarios differ only in the specified pressure (freshwater head) at the
landward boundary. The backend builds the Grid2D mesh internally for a rectangular
domain, so no mesh file needs to be created. The verbose solver output is suppressed.

In [ ]:
def run_scenario(scenario_index):
    params = ModelParameters()
    params.cellsize = CELLSIZE
    params.cellsize_x = CELLSIZE
    params.cellsize_y = CELLSIZE
    params.dt_max = DT_MAX
    params.coupled_relaxation = RELAX
    params.max_iterations = MAX_ITER
    params.min_iterations = min(MAX_ITER, params.min_iterations)
    params.total_time = TOTAL_TIME
    params.output_interval = TOTAL_TIME
    params.specified_pressure = ParameterRanges().specified_pressure_s[scenario_index]

    opts = ModelOptions()
    opts.backend = 'fipy'
    opts.save_vtk_files = False
    opts.save_vtk_files_all_steps = False
    opts.model_output_dir = OUTPUT_DIR

    # mesh file path is unused for a rectangular domain (Grid2D built internally)
    mesh_file = os.path.join(OUTPUT_DIR, 'mesh_nb_s%d.msh' % scenario_index)
    with open(os.devnull, 'w') as devnull, redirect_stdout(devnull):
        result = run_coupled_flow_model_fipy(params, opts, mesh_file)

    mesh = result[0]
    concentration = np.array(result[5])
    cell_centers = np.array(mesh._mesh.cellCenters)
    x = cell_centers[0]
    y = cell_centers[1]
    return x, y, concentration

## Helper functions: isochlor extraction and the Glover (1959) analytical solution

For each row of cells at constant elevation the horizontal position where the
concentration crosses a target isochlor is found by interpolation. The Glover (1959)
analytical fresh-salt interface is used as an independent reference.

In [ ]:
def extract_isochlor(x, y, concentration, target, n_bins=53):
    y_edges = np.linspace(y.min(), y.max(), n_bins)
    y_out = []
    x_out = []
    for i in range(len(y_edges) - 1):
        mask = (y >= y_edges[i]) & (y < y_edges[i + 1])
        if mask.sum() < 2:
            continue
        xs = x[mask]
        order = np.argsort(xs)
        xs = xs[order]
        cv = concentration[mask][order]
        if cv.max() >= target >= cv.min():
            y_out.append(0.5 * (y_edges[i] + y_edges[i + 1]))
            # reverse so the interpolation runs over increasing concentration
            x_out.append(np.interp(target, cv[::-1], xs[::-1]))
    return np.array(y_out), np.array(x_out)


def depth_sw_interface_glover1959(x, Q, K, rho_f, rho_s):
    gamma = (rho_s - rho_f) / rho_f
    y2 = 2.0 * Q / (gamma * K) * x + Q ** 2 / (gamma ** 2 * K ** 2)
    return np.sqrt(y2)

## Run the three steady state salt wedge scenarios

This is the expensive step (roughly 15 to 20 minutes total). The extracted isochlors are
cached so this cell can be skipped on a re-run. Delete the cache file to force a fresh run.

In [ ]:
if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, 'rb') as f:
        modeled = pickle.load(f)
    print('loaded cached isochlors from', CACHE_FILE)
else:
    modeled = []
    for k in range(3):
        t0 = time.time()
        x, y, concentration = run_scenario(k)
        isochlors = {}
        for frac in ISOCHLOR_FRACTIONS:
            isochlors[frac] = extract_isochlor(x, y, concentration, frac * MAX_CONC)
        modeled.append(isochlors)
        print('SS-%d done in %.0f s, max concentration %.4f'
              % (k + 1, time.time() - t0, concentration.max()))
    with open(CACHE_FILE, 'wb') as f:
        pickle.dump(modeled, f)

## Experimental data, escript reference and the analytical solution

The measured salt wedge positions are read from the benchmark table. The Glover (1959)
analytical interface uses the freshwater discharge of each experiment (the reported
fluxes include the 2.7 cm tank thickness). The escript reference results are read from
the published VTK files for comparison.

In [ ]:
df_exp = pd.read_csv(
    os.path.join(DATA_DIR, 'table_a1_steady_state_salt_wedge_locations.csv'))

Qs = np.array([1.42, 0.59, 1.19]) / 2.7 * 1e-4
K = 1050.0 / (24.0 * 60.0 * 60.0)
rho_f = 998.7
rho_s = 1026.0
xg = np.arange(0.0, 0.531, 0.001)
yg = [0.26 - depth_sw_interface_glover1959(xg, Q, K, rho_f, rho_s) for Q in Qs]

escript_files = [
    'benchmark_data/model_runs/runS0_specified_pressure_[0, 68.569]_final_output_Elements.vtu',
    'benchmark_data/model_runs/runS1_specified_pressure_[0, 19.591]_final_output_Elements.vtu',
    'benchmark_data/model_runs/runS2_specified_pressure_[0, 53.876]_final_output_Elements.vtu']

escript_iso = []
for f in escript_files:
    xy, conn, pt_names, pt_arrays, cell_names, cell_arrays = \
        lib.read_vtu_file.read_vtu_file(f)
    conc = np.array(pt_arrays[pt_names.index('concentration')])
    escript_iso.append(extract_isochlor(xy[:, 0], xy[:, 1], conc, 0.5 * MAX_CONC))

## Misfit between the modeled and experimental salt wedge

The modeled 0.5 isochlor is interpolated at the measured elevations and compared with the
measured horizontal positions. This is the same metric used in the original benchmark
comparison; the acceptance tolerance is about 4 cm. The escript reference misfit is shown
alongside for context.

In [ ]:
def misfit(model_y, model_x, exp_x, exp_y):
    modeled_at_exp = np.interp(exp_y, model_y, model_x)
    err = modeled_at_exp - exp_x
    return 100 * err.mean(), 100 * np.abs(err).mean(), 100 * np.sqrt((err ** 2).mean())

print('0.5 isochlor misfit versus experiment (cm)')
print('%-6s %18s %18s' % ('', 'FiPy', 'escript'))
print('%-6s %6s %5s %5s %6s %5s %5s'
      % ('', 'ME', 'MAE', 'RMSE', 'ME', 'MAE', 'RMSE'))
for k in range(3):
    exp_x = df_exp['x_ss%d' % (k + 1)].dropna().values / 100.0
    exp_y = df_exp['y_ss%d' % (k + 1)].dropna().values / 100.0
    my, mx = modeled[k][0.5]
    ey, ex = escript_iso[k]
    fme, fmae, frmse = misfit(my, mx, exp_x, exp_y)
    eme, emae, ermse = misfit(ey, ex, exp_x, exp_y)
    print('SS-%d   %6.2f %5.2f %5.2f %6.2f %5.2f %5.2f'
          % (k + 1, fme, fmae, frmse, eme, emae, ermse))

## Figure: modeled versus experimental salt wedge

The grey band is the modeled 0.1 to 0.9 isochlor (the fresh-salt mixing zone), the solid
line is the modeled 0.5 isochlor, the dotted line is the Glover (1959) analytical
interface, and the markers are the measured 0.5 isochlor positions for the three
experiments.

In [ ]:
markers = ['s', 'd', '^']

fig, ax = plt.subplots(figsize=(7, 4))
ax.set_xlabel('Distance (m)')
ax.set_ylabel('Elevation (m)')
ax.set_xlim(0.0, 0.53)
ax.set_ylim(0.0, 0.26)

leg_data = []
for k in range(3):
    y05, x05 = modeled[k][0.5]
    y01, x01 = modeled[k][0.1]
    y09, x09 = modeled[k][0.9]
    # interpolate the 0.1 and 0.9 isochlors onto the 0.5 isochlor elevations
    x01_i = np.interp(y05, y01, x01)
    x09_i = np.interp(y05, y09, x09)
    leg_band = ax.fill_betweenx(y05, x09_i, x01_i, edgecolor='grey',
                                facecolor='lightgrey', zorder=0)
    exp_x = df_exp['x_ss%d' % (k + 1)].dropna().values / 100.0
    exp_y = df_exp['y_ss%d' % (k + 1)].dropna().values / 100.0
    leg_pt = ax.scatter(exp_x, exp_y, marker=markers[k], facecolor='darkgray',
                        edgecolor='black', s=45)
    leg_data.append(leg_pt)
    leg_glover, = ax.plot(xg, yg[k], ls=':', color='black')
    leg_model, = ax.plot(x05, y05, color='black')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

legs = leg_data + [leg_glover, leg_model, leg_band]
labels = ['0.5 isochlor experiment %d' % i for i in range(1, 4)]
labels += ['analytical solution, Glover (1959)',
           'modeled 0.5 isochlor (FiPy)',
           'modeled 0.1 to 0.9 isochlor (FiPy)']
ax.legend(legs, labels, frameon=False, fontsize='small')

fig.tight_layout()
fig.savefig(os.path.join(DATA_DIR, 'model_vs_experimental_salt_wedge_fipy.png'), dpi=150)
fig.savefig(os.path.join(DATA_DIR, 'model_vs_experimental_salt_wedge_fipy.pdf'))
print('saved figure to', os.path.join(DATA_DIR, 'model_vs_experimental_salt_wedge_fipy.png'))